<a href="https://colab.research.google.com/github/LNDOTIS/FlyRank-AI-Internship---Machine-Learning/blob/main/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

# Capstone — Predicting Near-Term Organic Search Decline

## Research question

Can historical search-performance and content-lifecycle signals be used to rank content pages that are more likely to experience near-term organic-search decline?

The decision this work supports is **which pages a content team should review first**.

The unit of analysis is a **content page**. Each page receives a risk score and is placed in a ranked review queue. The queue is intended to support human decision-making rather than automatically change content.

The central idea is simple:

> Pages with substantial historical search visibility, weaker historical performance, and/or lifecycle signals may deserve earlier review when the model assigns them a higher measured risk.

The work deliberately treats the output as **decision-support**. A high score means that a page is worth reviewing earlier; it does not establish why the page is declining or that a particular SEO intervention will improve performance.

## Research question

**How well can a simple Logistic Regression model rank pages for near-term organic-search decline compared with a transparent rule-based baseline?**

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
NumPy: 2.0.2
Pandas: 2.2.2


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## Data source

This project uses the **FlyRank pseudonymized warehouse release v20260703**.

The warehouse contains:

- `dim_clients`
- `dim_content`
- `fact_content_daily_performance`
- `fact_content_query_90d`

The analysis uses the daily content-performance data to construct page-level historical features.

The modeling frame uses a **March 2026 feature window** and an **April 2026 future window**.

This separation is important: the features describe information available before the prediction period, while the target is defined from the subsequent period.

### Main features

The final Logistic Regression model uses:

- `imp_prev30`
- `clk_prev30`
- `avg_position_prev30`
- `days_with_impressions`
- `content_age_days`

Pseudonymous identifiers such as `client_hash_id` and `content_hash_id` are retained for grouping and review, but are **not used as model features**.

### Deliberate exclusions

The following were excluded because they could leak the target or encode an existing decision:

- `is_declining`
- `trend_direction`
- `trend_pct`
- future-window performance variables
- FlyRank product flags or scores
- client/content identifiers as predictive features

The target is:

> `is_declining = 1` when future impressions are more than 20% below the previous 30-day level.

The dataset contains 100,893 modeled observations and an observed decline rate of approximately 51.48%.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## Method

The model is Logistic Regression.

Logistic Regression was selected because this is a binary prediction/ranking problem and the goal is not simply to maximize complexity. A linear probabilistic model provides a relatively interpretable first learned model that can be compared directly with the transparent Week-4 rule.

The baseline and model are evaluated as rankings because the operational question is:

> Which pages should be reviewed first?

Therefore, precision@K is especially useful.

### Baseline

The Week-4 baseline is a transparent hand-written rule. It ranks pages using historical visibility and lifecycle conditions rather than fitted model weights.

The baseline is intentionally simple and human-readable. It provides a floor that the learned model must beat on the same evaluation data.

### Validation

The initial Week-5 comparison used the same modeling frame and evaluation protocol for the baseline and Logistic Regression.

A grouped-client validation was subsequently used as a more conservative check because pages belonging to the same client can share hidden characteristics. Grouping prevents pages from the same client appearing across training and test sets.

### Leakage checks

The final feature set was audited for:

1. label-derived variables;
2. future-window variables;
3. overlapping target windows;
4. existing product decisions or flags;
5. pseudonymous identifiers used as predictive variables.

The model therefore uses only signals intended to be available before the target period.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## Evaluation metric

Precision@K answers the operational question directly:

> Of the K pages placed at the top of the review queue, what fraction were actually labeled as declining?

The base rate is reported alongside precision because a high precision can otherwise be misleading when the target itself is common.

The baseline and Logistic Regression use the **same test rows** and the **same target labels**.



---


| Metric       | Baseline | Logistic Regression |
|--------------|----------|---------------------|
| Precision@20 | 0.35     | 0.60                |
| Precision@50 | 0.38     | 0.46                |
| Precision@100| 0.36     | 0.54                |
| ROC-AUC      | —        | 0.62                |

## 5. Limitations

*What this work cannot claim.*

## Limitations and honest framing

Several limitations constrain what this analysis can claim.

First, the model identifies statistical patterns associated with near-term decline in this dataset. It does not establish that refreshing a page will cause its search performance to recover.

Second, the model is trained and evaluated on a particular set of pseudonymized clients and content pages. Performance may differ for a new client or a different period.

Third, historical search performance is not the same as a diagnosis of why a page declined. A high model score indicates that a page is worth reviewing earlier; it does not identify the causal reason for the decline.

Fourth, the target is operationally defined as a measured reduction in future impressions. This is useful for ranking review candidates, but it is only a proxy for the broader business concept of "content health."

Finally, the ranked queue should not be interpreted as an automated content-editing system. Human review remains necessary before any refresh, rewrite, consolidation, redirect, or technical intervention.

The appropriate interpretation is therefore:

> **The model provides directional decision-support for prioritizing manual review of pages that show a higher measured risk of near-term search decline.**

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The model output is converted into a human-reviewed action queue.

### 1. High-risk + high-exposure

**Reason code:** `high_risk_high_exposure`

**Action:** Prioritize for manual performance review.

These pages combine a higher model score with substantial historical search exposure.

Human question:

> If this page is genuinely at risk, is the potential value of reviewing it high enough to justify the review cost?

---

### 2. High-risk + stale + visible

**Reason code:** `high_risk_stale_visible`

**Action:** Review freshness and content relevance.

Human questions:

- Is the information still accurate?
- Has the topic changed?
- Does the page still match the search intent?
- Is the historical exposure meaningful enough to justify a refresh review?

---

### 3. High-risk + weak position

**Reason code:** `high_risk_position`

**Action:** Review search intent, relevance and ranking context.

Human questions:

- Has the search landscape changed?
- Does the page still match the intended query?
- Are there technical or indexing issues?
- Are competitors or SERP features changing the opportunity?

---

### 4. High-risk general review

**Reason code:** `high_risk_general_review`

**Action:** Broader manual review.

This category is used when the model assigns a higher risk score but no single diagnostic signal is dominant enough to justify a more specific action.

This is deliberately a fallback category. It means "review this page" rather than "we know why this page is declining."

---

## What should not be automated

The system should not automatically:

- rewrite content;
- delete pages;
- redirect URLs;
- change search intent;
- change internal-link structures;
- declare a page technically broken;
- conclude that a specific SEO intervention caused or will cause recovery.

Those decisions require human review and additional evidence.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The deployed paper uses a small number of artifacts rather than reproducing the entire notebook.

The main artifacts are:

1. Model-vs-baseline precision@K table.
2. Precision@K comparison chart.
3. Feature interpretation chart.
4. Ranked action queue.

All exported artifacts are generated from this notebook so that the paper can be traced back to a reproducible analysis run.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.